In [ ]:
# Some code in this file was generated using AI

import pyreadstat
import pandas as pd
import numpy as np

file_path = "h22f2a.dta"

# 1. Target columns to extract
cols_to_pull = [
    "hhidpn",  # Unique Person ID
    "sa019",  # Age calculation
    "sb014",  # Years of education (0 - 17)
    "sb017m",  # Highest degree masked
    "sj3578",  # Consider self retired
    "sj3581",  # Year retired
    "sd174",  # Immediate recall score (phone/in-person)
    "sd174w",  # Immediate recall score (web)
    "sd184",  # Delayed recall score (phone/in-person)
    "sd184w",  # Delayed recall score (web)
]

print("Reading selected columns from h22f2a.dta...")
df, meta = pyreadstat.read_dta(file_path, usecols=cols_to_pull)

print(f"Loaded {len(df):,} total survey respondents.")

# 2. Combine Phone/In-Person + Web Recall Scores (0 to 10 each)
immediate = df["sd174"].combine_first(df["sd174w"])
delayed = df["sd184"].combine_first(df["sd184w"])

# Total Memory Recall (0 to 20 scale)
df["memory_score"] = immediate + delayed

# 3. Clean Age and filter to aging population (ages 50 to 90)
df["age"] = pd.to_numeric(df["sa019"], errors="coerce")
df = df.dropna(subset=["age", "memory_score"]).copy()
df = df[(df["age"] >= 50) & (df["age"] <= 90)].copy()


# 4. Map Retirement Status (sj3578)
# In HRS: 1 = Completely retired, 2 = Partly retired, 3 = Not retired
# (Check value labels or map standard HRS codes)
def map_retirement(val):
    if val == 1:
        return "Completely Retired"
    elif val == 2:
        return "Partly Retired"
    elif val == 3:
        return "Not Retired"
    else:
        return "Not Retired"


df["work_status"] = df["sj3578"].apply(map_retirement)

# 5. Calculate Years Since Retirement (Survey Year is 2022)
# Positive = Years already retired; 0 = Just retired; NaN = Still working
retire_year = pd.to_numeric(df["sj3581"], errors="coerce")
df["years_retired"] = np.where(
    (df["work_status"] == "Completely Retired")
    & (retire_year >= 1970)
    & (retire_year <= 2022),
    2022 - retire_year,
    np.nan,
)


# 6. Map Education Tier from Years of Schooling (sb014)
def map_education(years):
    if pd.isna(years) or years < 0:
        return "High School / Vocational"
    if years < 12:
        return "Less than High School"
    elif years <= 12:
        return "High School / GED"
    elif years <= 15:
        return "Some College / Associate"
    else:
        return "College Degree (BA/BS+)"


df["education"] = df["sb014"].apply(map_education)

# 7. Build the final clean DataFrame
clean_df = pd.DataFrame(
    {
        "id": df["hhidpn"].astype(str),
        "age": df["age"].astype(int),
        "work_status": df["work_status"],
        "years_retired": df["years_retired"].round(1),
        "memory_score": df["memory_score"].round(1),
        "immediate_recall": immediate.loc[df.index].round(1),
        "delayed_recall": delayed.loc[df.index].round(1),
        "education": df["education"],
    }
)

# Save to CSV
output_file = "clean_memory_retirement.csv"
clean_df.to_csv(output_file, index=False)

print(f"\nCreated '{output_file}' with {len(clean_df):,} rows.")
print("\nPreview of your clean dashboard dataset:")
print(clean_df.head(10))
print("\nWork Status distribution:")
print(clean_df["work_status"].value_counts())
print("\nEducation distribution:")
print(clean_df["education"].value_counts())

Reading selected columns from h22f2a.dta...
Loaded 15,787 total survey respondents.

Created 'clean_memory_retirement.csv' with 14,350 rows.

Preview of your clean dashboard dataset:
          id  age         work_status  years_retired memory_score  \
0   10001010   83  Completely Retired            NaN            4   
1   10003030   66  Completely Retired           30.0            7   
2   10004040   77  Completely Retired            NaN           12   
3   10013040   74  Completely Retired            NaN           11   
4   10038010   85  Completely Retired            NaN            8   
5   10038040   79  Completely Retired            NaN           10   
6   10059020   86  Completely Retired            0.0            3   
8   10075020   86  Completely Retired            NaN            1   
9   10106020   89  Completely Retired            NaN            9   
11  10299010   81  Completely Retired            NaN           10   

   immediate_recall delayed_recall                 educat